In [14]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
/kaggle/input/data-sider-raww/meddra_all_se.tsv
/kaggle/input/drugbank-5-0/full database.xml
/kaggle/input/datasets/vungocthien/dataset-dont-split/Final_Dataset_DDI.csv


---

# Tái thực nghiệm drugbank 5.0

In [15]:
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
import os
import requests
import time
from lxml import etree
from sklearn.metrics.pairwise import cosine_similarity
from typing import Optional

# I. Trích xuất dữ liệu từ Drugbank

In [16]:
class DrugBankParser:
    """
    Class: đọc, trích xuất và lọc dữ liệu từ file XML của DrugBank.
    Kết quả trả về là Nodes (thông tin thuốc) và Edges (tương tác thuốc).
    """
    def __init__(self, xml_path):
        self.xml_path = xml_path
        self.ns = {'db': 'http://www.drugbank.ca'}
        self.df_nodes = None
        self.df_edges = None

    def _get_chemical_info(self, drug_element):
        """Trích xuất ID, Tên, SMILES và InChIKey."""
        try:
            drug_id = drug_element.find("db:drugbank-id[@primary='true']", self.ns).text
            name = drug_element.find("db:name", self.ns).text
        except AttributeError:
            return None

        smiles = None
        inchikey = None 

        for prop in drug_element.findall("db:calculated-properties/db:property", self.ns):
            kind = prop.find("db:kind", self.ns).text
            val = prop.find("db:value", self.ns).text
            if kind == "SMILES":
                smiles = val
            elif kind == "InChIKey":
                inchikey = val
        
        if not smiles or not inchikey:
            return None

        return {
            'drugbank_id': drug_id,
            'name': name,
            'smiles': smiles,
            'inchikey': inchikey
        }

    def _get_atc_codes(self, drug_element):
        """Lấy danh sách mã ATC Level 1 theo phương pháp của bài báo[cite: 1173, 1174]."""
        atc_list = []
        for atc in drug_element.findall("db:atc-codes/db:atc-code", self.ns):
            code = atc.get('code')
            if code:
                atc_list.append(code[0]) # Chỉ lấy ký tự đầu đại diện cho anatomical main group
        return list(set(atc_list))

    def _get_mesh_terms(self, drug_element):
        """Lấy danh sách MeSH ID để tính toán độ tương đồng ngữ nghĩa sau này[cite: 1195]."""
        mesh_list = []
        for cat in drug_element.findall("db:categories/db:category", self.ns):
            mesh_id_elem = cat.find("db:mesh-id", self.ns)
            if mesh_id_elem is not None and mesh_id_elem.text:
                mesh_list.append(mesh_id_elem.text)
        return mesh_list

    def _get_interactions(self, drug_element, current_drug_id):
        """Trích xuất các cạnh (edges) tương tác trong mạng lưới[cite: 1071]."""
        edges = []
        for interaction in drug_element.findall("db:drug-interactions/db:drug-interaction", self.ns):
            target_id = interaction.find("db:drugbank-id", self.ns).text
            if target_id:
                edges.append({
                    'source': current_drug_id,
                    'target': target_id
                })
        return edges

    def parse(self):
        """Hàm thực thi chính để duyệt qua toàn bộ XML và trả về kết quả."""
        if not os.path.exists(self.xml_path):
            raise FileNotFoundError(f"Không tìm thấy file tại: {self.xml_path}")

        print(f"--- Đang bắt đầu xử lý DrugBank XML ---")
        tree = ET.parse(self.xml_path)
        root = tree.getroot()
        
        nodes = []
        all_edges = []
        
        for drug in root.findall('db:drug', self.ns):
            # 1. Lấy thông tin cơ bản & hóa học
            chem_info = self._get_chemical_info(drug)
            if chem_info is None: continue
                
            # 2. Lấy ATC (Yêu cầu bắt buộc để tính toán semantic sau này)
            atc = self._get_atc_codes(drug)
            if not atc: continue 
                
            # 3. Lấy MeSH (Yêu cầu bắt buộc theo bài báo)
            mesh = self._get_mesh_terms(drug)
            if not mesh: continue 
            
            # Gộp thông tin vào Node
            node_data = chem_info.copy()
            node_data['atc_codes'] = atc
            node_data['mesh_terms'] = mesh
            nodes.append(node_data)
            
            # 4. Lấy Tương tác (Cạnh)
            drug_interactions = self._get_interactions(drug, chem_info['drugbank_id'])
            all_edges.extend(drug_interactions)

        self.df_nodes = pd.DataFrame(nodes)
        
        # 5. Lọc cạnh: Chỉ giữ lại tương tác giữa các thuốc có đủ thông tin (Nodes hợp lệ)
        valid_ids = set(self.df_nodes['drugbank_id'])
        self.df_edges = pd.DataFrame(all_edges)
        self.df_edges = self.df_edges[self.df_edges['target'].isin(valid_ids)]
        
        # Loại bỏ tương tác trùng lặp (vì bài báo coi đây là mạng vô hướng )
        self.df_edges = self.df_edges.drop_duplicates()

        print(f"Xử lý xong!")
        print(f"Số lượng thuốc (Nodes): {len(self.df_nodes)}")
        print(f"Số lượng tương tác (Edges): {len(self.df_edges)}")
        
        return self.df_nodes, self.df_edges

    def save_to_csv(self, nodes_path='drug_nodes.csv', edges_path='drug_edges.csv'):
        if self.df_nodes is not None and self.df_edges is not None:
            self.df_nodes.to_csv(nodes_path, index=False)
            self.df_edges.to_csv(edges_path, index=False)
            print(f"Đã lưu dữ liệu vào {nodes_path} và {edges_path}")

parser = DrugBankParser('/kaggle/input/drugbank-5-0/full database.xml')
df_nodes, df_edges = parser.parse()
parser.save_to_csv()

--- Đang bắt đầu xử lý DrugBank XML ---
Xử lý xong!
Số lượng thuốc (Nodes): 1912
Số lượng tương tác (Edges): 390300
Đã lưu dữ liệu vào drug_nodes.csv và drug_edges.csv


In [17]:
node = pd.read_csv('/kaggle/working/drug_nodes.csv')
node.head()

,drugbank_id,name,smiles,inchikey,atc_codes,mesh_terms
0,DB00006,Bivalirudin,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...,OIRCOABEOLEUMC-GEJPAHFPSA-N,['B'],"['D000602', 'D000925', 'D058833', 'D000991', '..."
1,DB00014,Goserelin,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...,BLCLNMBMMGCOAS-URPVMXJPSA-N,['L'],"['D000602', 'D000970', 'D018931', 'D020164', '..."
2,DB00035,Desmopressin,NC(=O)CC[C@@H]1NC(=O)[C@H](CC2=CC=CC=C2)NC(=O)...,NFLWUMRGJYTJIN-NXBWRCJVSA-N,['H'],"['D000602', 'D050034', 'D001127', 'D002317', '..."
3,DB00050,Cetrorelix,CC(C)C[C@H](NC(=O)[C@@H](CCCNC(N)=O)NC(=O)[C@H...,SBNPWPIBESPSIF-MHWMIDJBSA-N,['H'],"['D000602', 'D020164', 'D005299', 'D005300', '..."
4,DB00080,Daptomycin,CCCCCCCCCC(=O)N[C@@H](CC1=CNC2=C1C=CC=C2)C(=O)...,DOAKLVKFURWEDJ-RWDRXURGSA-N,['J'],"['D000602', 'D000900', 'D000890', 'D020164', '..."


In [15]:
edges = pd.read_csv('/kaggle/working/drug_edges.csv')
edges.tail()

,source,target
390295,DB13143,DB04876
390296,DB13143,DB00126
390297,DB13143,DB00582
390298,DB13143,DB00682
390299,DB13143,DB04898


---

In [16]:
sider = pd.read_csv('/kaggle/input/data-sider-raww/meddra_all_se.tsv', sep = '\t', header=None)
sider.tail(5)

,0,1,2,3,4,5
309844,CID171306834,CID071306834,C3203358,PT,C1145670,Respiratory failure
309845,CID171306834,CID071306834,C3665386,LLT,C3665386,Abnormal vision
309846,CID171306834,CID071306834,C3665386,PT,C3665347,Visual impairment
309847,CID171306834,CID071306834,C3665596,LLT,C3665596,Warts
309848,CID171306834,CID071306834,C3665596,PT,C0347390,Skin papilloma


In [17]:
import pandas as pd
import requests
import time
import numpy as np
from lxml import etree

In [18]:
class DDI_NetworkFilter:
    def __init__(self, nodes_csv_path, edges_csv_path):
        """
        INPUT: 
        - nodes_csv_path: Đường dẫn file node (đã có SMILES, ATC, MeSH) 
        - edges_csv_path: Đường dẫn file cạnh tương tác
        """
        self.nodes_path = nodes_csv_path
        self.edges_path = edges_csv_path
        self.df_nodes = None
        self.df_edges = None

    def load_data(self):
        """Hàm nạp dữ liệu từ các file CSV của bước 1."""
        self.df_nodes = pd.read_csv(self.nodes_path)
        self.df_edges = pd.read_csv(self.edges_path)
        print(f" Input: {len(self.df_nodes)} nodes và {len(self.df_edges)} edges.")

    def filter_and_standardize(self):
        """
        THỰC THI LỌC:
        1. Giữ cạnh có cả 2 node nằm trong danh sách node hợp lệ[cite: 1101].
        2. Chuẩn hóa vô hướng (A-B và B-A là một)[cite: 1067].
        3. Loại bỏ node cô lập (không có tương tác nào)[cite: 1106].
        """
        # 1. Lọc cạnh theo Node
        valid_ids = set(self.df_nodes['drugbank_id'])
        mask = (self.df_edges['source'].isin(valid_ids)) & (self.df_edges['target'].isin(valid_ids))
        df_clean_edges = self.df_edges[mask].copy()

        # 2. Xử lý đối xứng & Xóa trùng
        # Sắp xếp để đảm bảo source < target về ký tự
        edge_vals = df_clean_edges[['source', 'target']].values
        df_clean_edges[['source', 'target']] = np.sort(edge_vals, axis=1)
        df_clean_edges = df_clean_edges.drop_duplicates(subset=['source', 'target'])
        
        # Loại bỏ self-loop (thuốc tương tác với chính nó)
        df_clean_edges = df_clean_edges[df_clean_edges['source'] != df_clean_edges['target']]
        self.df_edges = df_clean_edges

        # 3. Lọc ngược Node: Chỉ giữ lại các thuốc có trong danh sách tương tác sạch
        active_ids = set(self.df_edges['source']).union(set(self.df_edges['target']))
        self.df_nodes = self.df_nodes[self.df_nodes['drugbank_id'].isin(active_ids)].copy()

        print(f" Kết quả: {len(self.df_nodes)} nodes và {len(self.df_edges)} edges.")

    def get_clean_data(self):
        """
        OUTPUT: Trả về 2 DataFrame sạch để dùng cho các bước sau.
        """
        return self.df_nodes, self.df_edges

    def save_to_csv(self, nodes_out, edges_out):
        """Lưu kết quả ra file mới."""
        self.df_nodes.to_csv(nodes_out, index=False)
        self.df_edges.to_csv(edges_out, index=False)

nodes_step1 = '/kaggle/working/drug_nodes.csv'
edges_step1 = '/kaggle/working/drug_edges.csv'

processor = DDI_NetworkFilter(nodes_step1, edges_step1)

processor.load_data()
processor.filter_and_standardize()
processor.save_to_csv('drug_node_ver2.csv', 'drug_edge_final.csv')

 Input: 1912 nodes và 390300 edges.
 Kết quả: 1608 nodes và 195150 edges.


In [19]:
class SIDER_DrugMapper:
    """
    Class thực hiện ánh xạ thuốc từ DrugBank với dữ liệu tác dụng phụ từ SIDER
    và gom nhóm chúng thành dạng danh sách (List) cho từng thuốc.
    """
    def __init__(self, drug_nodes_path, sider_tsv_path):
        self.drug_nodes_path = drug_nodes_path
        self.sider_path = sider_tsv_path
        
        self.df_drugs = None
        self.df_sider_raw = None
        self.df_mapped_pairs = None
        self.df_nodes_with_se = None

    def _stitch_stereo_to_pubchem(self, cid_str):
        """Chuyển đổi ID STITCH sang PubChem CID."""
        if isinstance(cid_str, str) and cid_str.startswith('CID'):
            return int(cid_str[3:])
        return None

    def _get_cid_from_inchikey(self, inchikey):
        """Lấy PubChem CID từ InChIKey thông qua API PubChem."""
        try:
            url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/inchikey/{inchikey}/XML"
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                root = etree.XML(response.content)
                ns = {"atom": "http://www.ncbi.nlm.nih.gov"}
                cids = root.xpath("//atom:PC-Compound/atom:PC-Compound_id/atom:PC-CompoundType/atom:PC-CompoundType_id/atom:PC-CompoundType_id_cid", namespaces=ns)
                if cids:
                    return int(cids[0].text)
        except Exception as e:
            print(f"  [!] Lỗi InChIKey {inchikey}: {e}")
        return None

    def _get_cid_from_name(self, drug_id):
        """Hàm dự phòng tìm CID bằng DrugBank ID."""
        try:
            url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{drug_id}/cids/JSON"
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                data = response.json()
                return data['IdentifierList']['CID'][0]
        except:
            pass
        return None

    def load_and_map_drugs(self):
        """Giai đoạn 1: Nạp thuốc và tìm PubChem CID tương ứng."""
        print(f"[*] 1. Đang nạp danh sách thuốc và tìm CID...")
        self.df_drugs = pd.read_csv(self.drug_nodes_path)
        
        # Đảm bảo có cột drugbank_id
        if 'drugbank_id' not in self.df_drugs.columns and 'id' in self.df_drugs.columns:
            self.df_drugs.rename(columns={'id': 'drugbank_id'}, inplace=True)
        
        cids = []
        for index, row in self.df_drugs.iterrows():
            cid = None
            if 'inchikey' in row and pd.notna(row['inchikey']):
                cid = self._get_cid_from_inchikey(row['inchikey'])
            if cid is None:
                cid = self._get_cid_from_name(row['drugbank_id'])
            
            cids.append(cid)
            if index % 50 == 0: print(f"  - Đã xử lý {index}/{len(self.df_drugs)} thuốc...")
            time.sleep(0.2)

        self.df_drugs['pubchem_cid'] = cids
        self.df_drugs = self.df_drugs.dropna(subset=['pubchem_cid'])
        self.df_drugs['pubchem_cid'] = self.df_drugs['pubchem_cid'].astype(int)

    def load_sider_data(self):
        """Giai đoạn 2: Nạp dữ liệu tác dụng phụ từ file TSV của SIDER[cite: 1201]."""
        print(f"[*] 2. Đang nạp dữ liệu SIDER từ: {self.sider_path}...")
        columns = ['stitch_id_flat', 'stitch_id_sterio', 'umls_cui_label', 'meddra_type', 'umls_cui_meddra', 'side_effect_name']
        self.df_sider_raw = pd.read_csv(self.sider_path, sep='\t', names=columns)
        
        self.df_sider_raw['pubchem_cid'] = self.df_sider_raw['stitch_id_sterio'].apply(self._stitch_stereo_to_pubchem)
        self.df_sider_raw = self.df_sider_raw[['pubchem_cid', 'umls_cui_meddra', 'side_effect_name']]

    def process_mapping_to_list(self):
        """Giai đoạn 3: Ghép nối và gom nhóm tác dụng phụ thành định dạng List."""
        print("[*] 3. Đang ghép nối và gom nhóm tác dụng phụ thành dạng List...")
        
        # Merge dựa trên PubChem CID
        df_merged = self.df_drugs.merge(self.df_sider_raw, on='pubchem_cid', how='inner')
        
        # Gom nhóm theo drugbank_id và tạo list các mã UMLS CUI
        sider_grouped = df_merged.groupby('drugbank_id')['umls_cui_meddra'].apply(lambda x: list(set(x))).reset_index()
        sider_grouped.rename(columns={'umls_cui_meddra': 'side_effect_codes'}, inplace=True)
        
        # Ghép ngược lại vào bảng nodes ban đầu (dùng left join để giữ lại thuốc không có tác dụng phụ)
        self.df_nodes_with_se = pd.merge(self.df_drugs, sider_grouped, on='drugbank_id', how='left')
        
        # Xử lý các giá trị NaN (thuốc không có tác dụng phụ) thành list rỗng
        self.df_nodes_with_se['side_effect_codes'] = self.df_nodes_with_se['side_effect_codes'].apply(
            lambda x: x if isinstance(x, list) else []
        )
        
        count_have_se = self.df_nodes_with_se['side_effect_codes'].apply(len).gt(0).sum()
        print(f" -> Hoàn tất: {count_have_se}/{len(self.df_nodes_with_se)} thuốc có dữ liệu tác dụng phụ.")

    def save_output(self, output_path):
        """OUTPUT: Lưu bảng Nodes kèm danh sách tác dụng phụ ra file CSV."""
        if self.df_nodes_with_se is not None:
            self.df_nodes_with_se.to_csv(output_path, index=False)
            print(f"[V] Kết quả cuối cùng đã được lưu tại: {output_path}")



NODE_Path = '/kaggle/working/drug_node_ver2.csv' 
SIDER_Path = '/kaggle/input/data-sider-raww/meddra_all_se.tsv'
    
mapper = SIDER_DrugMapper(NODE_Path, SIDER_Path)
    
mapper.load_and_map_drugs()    
mapper.load_sider_data()       
mapper.process_mapping_to_list() 
mapper.save_output('drug_nodes_with_side_effects.csv')

[*] 1. Đang nạp danh sách thuốc và tìm CID...
  - Đã xử lý 0/1608 thuốc...
  - Đã xử lý 50/1608 thuốc...
  - Đã xử lý 100/1608 thuốc...
  - Đã xử lý 150/1608 thuốc...
  - Đã xử lý 200/1608 thuốc...
  - Đã xử lý 250/1608 thuốc...
  - Đã xử lý 300/1608 thuốc...
  - Đã xử lý 350/1608 thuốc...
  - Đã xử lý 400/1608 thuốc...
  - Đã xử lý 450/1608 thuốc...
  - Đã xử lý 500/1608 thuốc...
  - Đã xử lý 550/1608 thuốc...
  - Đã xử lý 600/1608 thuốc...
  - Đã xử lý 650/1608 thuốc...
  - Đã xử lý 700/1608 thuốc...
  - Đã xử lý 750/1608 thuốc...
  - Đã xử lý 800/1608 thuốc...
  - Đã xử lý 850/1608 thuốc...
  - Đã xử lý 900/1608 thuốc...
  - Đã xử lý 950/1608 thuốc...
  - Đã xử lý 1000/1608 thuốc...
  - Đã xử lý 1050/1608 thuốc...
  - Đã xử lý 1100/1608 thuốc...
  - Đã xử lý 1150/1608 thuốc...
  - Đã xử lý 1200/1608 thuốc...
  - Đã xử lý 1250/1608 thuốc...
  - Đã xử lý 1300/1608 thuốc...
  - Đã xử lý 1350/1608 thuốc...
  - Đã xử lý 1400/1608 thuốc...
  - Đã xử lý 1450/1608 thuốc...
  - Đã xử lý 1500

In [20]:
a = pd.read_csv('/kaggle/working/drug_nodes_with_side_effects.csv')
a.head(10)

,drugbank_id,name,smiles,inchikey,atc_codes,mesh_terms,pubchem_cid,side_effect_codes
0,DB00006,Bivalirudin,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...,OIRCOABEOLEUMC-GEJPAHFPSA-N,['B'],"['D000602', 'D000925', 'D058833', 'D000991', '...",16129704,"['C0035126', 'C0002962', 'C0042075', 'C0917801..."
1,DB00014,Goserelin,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...,BLCLNMBMMGCOAS-URPVMXJPSA-N,['L'],"['D000602', 'D000970', 'D018931', 'D020164', '...",5311128,[]
2,DB00035,Desmopressin,NC(=O)CC[C@@H]1NC(=O)[C@H](CC2=CC=CC=C2)NC(=O)...,NFLWUMRGJYTJIN-NXBWRCJVSA-N,['H'],"['D000602', 'D050034', 'D001127', 'D002317', '...",16051933,[]
3,DB00080,Daptomycin,CCCCCCCCCC(=O)N[C@@H](CC1=CNC2=C1C=CC=C2)C(=O)...,DOAKLVKFURWEDJ-RWDRXURGSA-N,['J'],"['D000602', 'D000900', 'D000890', 'D020164', '...",16134395,[]
4,DB00091,Cyclosporine,CC[C@@H]1NC(=O)[C@H]([C@H](O)[C@H](C)C\C=C\C)N...,PMATZTZNYRCHOR-CGLBZJNRSA-N,"['L', 'S']","['D000602', 'D000890', 'D000935', 'D018501', '...",5284373,[]
5,DB00104,Octreotide,[H][C@]1(NC(=O)[C@H](CCCCN)NC(=O)[C@@H](CC2=CN...,DEQANNDTNATYII-OULOTJBUSA-N,['H'],"['D000602', 'D000970', 'D018931', 'D020164', '...",448601,[]
6,DB00115,Cyanocobalamin,[C@H]1([C@@H]2[C@H]([C@@H](N3C4=CC(=C(C)C=C4[N...,RMRCNWBMXRMIRW-WZHZPDAFSA-L,['B'],"['D001393', 'D020164', 'D045728', 'D000066888'...",166596686,[]
7,DB00122,Choline,C[N+](C)(C)CCO,OEYIOHPDSNJKLS-UHFFFAOYSA-N,['N'],"['D000146', 'D000438', 'D000588', 'D000605', '...",305,[]
8,DB00126,Vitamin C,[H][C@@]1(OC(=O)C(O)=C1O)[C@@H](O)CO,CIWBSHSKHKDKBQ-JLAZNSOCSA-N,"['G', 'A', 'S']","['D000144', 'D000975', 'D002241', 'D002264', '...",54670067,[]
9,DB00130,L-Glutamine,N[C@@H](CCC(N)=O)C(O)=O,ZDXPYRJPNDTMRX-VKHMYHEASA-N,['A'],"['D000596', 'D024361', 'D000599', 'D021542', '...",5961,"['C0917801', 'C0011603', 'C0008031', 'C0010346..."


In [22]:
b = a.drop(columns = ['inchikey', 'pubchem_cid'])
b.to_csv('drug_node_final.csv')
b.head(10)

,drugbank_id,name,smiles,atc_codes,mesh_terms,side_effect_codes
0,DB00006,Bivalirudin,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...,['B'],"['D000602', 'D000925', 'D058833', 'D000991', '...","['C0035126', 'C0002962', 'C0042075', 'C0917801..."
1,DB00014,Goserelin,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...,['L'],"['D000602', 'D000970', 'D018931', 'D020164', '...",[]
2,DB00035,Desmopressin,NC(=O)CC[C@@H]1NC(=O)[C@H](CC2=CC=CC=C2)NC(=O)...,['H'],"['D000602', 'D050034', 'D001127', 'D002317', '...",[]
3,DB00080,Daptomycin,CCCCCCCCCC(=O)N[C@@H](CC1=CNC2=C1C=CC=C2)C(=O)...,['J'],"['D000602', 'D000900', 'D000890', 'D020164', '...",[]
4,DB00091,Cyclosporine,CC[C@@H]1NC(=O)[C@H]([C@H](O)[C@H](C)C\C=C\C)N...,"['L', 'S']","['D000602', 'D000890', 'D000935', 'D018501', '...",[]
5,DB00104,Octreotide,[H][C@]1(NC(=O)[C@H](CCCCN)NC(=O)[C@@H](CC2=CN...,['H'],"['D000602', 'D000970', 'D018931', 'D020164', '...",[]
6,DB00115,Cyanocobalamin,[C@H]1([C@@H]2[C@H]([C@@H](N3C4=CC(=C(C)C=C4[N...,['B'],"['D001393', 'D020164', 'D045728', 'D000066888'...",[]
7,DB00122,Choline,C[N+](C)(C)CCO,['N'],"['D000146', 'D000438', 'D000588', 'D000605', '...",[]
8,DB00126,Vitamin C,[H][C@@]1(OC(=O)C(O)=C1O)[C@@H](O)CO,"['G', 'A', 'S']","['D000144', 'D000975', 'D002241', 'D002264', '...",[]
9,DB00130,L-Glutamine,N[C@@H](CCC(N)=O)C(O)=O,['A'],"['D000596', 'D024361', 'D000599', 'D021542', '...","['C0917801', 'C0011603', 'C0008031', 'C0010346..."


---

# 1. ATC

In [23]:
import pandas as pd
import numpy as np
import ast
import os

In [24]:
class ATCFeatureGenerator:
    """
    TÍNH TOÁN ĐẶC TRƯNG ATC (Drug therapeutic-based similarity)
    - Input: File CSV chứa cột 'atc_codes' dạng list (level 1).
    - Output: Ma trận tương đồng Cosine Similarity.
    """
    def __init__(self, input_path):
        if not os.path.exists(input_path):
            raise FileNotFoundError(f"Không tìm thấy file {input_path}")
        
        self.df = pd.read_csv(input_path)
        self.drug_ids = self.df['drugbank_id'].tolist()
        self.N = len(self.drug_ids)
        
        # Chuyển chuỗi văn bản thành danh sách Python thực tế
        self.df['atc_codes'] = self.df['atc_codes'].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else []
        )

    def compute_atc_similarity(self, output_path='feature_ATC.csv'):
        print(f"Bắt đầu tính toán đặc trưng ATC cho {self.N} loại thuốc...")
        
        # 1. Tạo bộ từ vựng ATC (Vocabulary)
        unique_atc = set()
        for codes in self.df['atc_codes']:
            unique_atc.update({str(c) for c in codes if str(c)})
        
        vocab = sorted(list(unique_atc))
        vocab_index = {code: i for i, code in enumerate(vocab)}
        n_features = len(vocab)
        print(f"  -> Tìm thấy {n_features} mã ATC Level 1 duy nhất.")
        print(f"[1] Vocabulary (14 cụm): {vocab}")
        print("*"*30)

        # 2. Xây dựng ma trận TF (Binary Term Frequency)
        # Mỗi hàng đại diện cho 1 thuốc, mỗi cột là 1 mã ATC 
        tf_matrix = np.zeros((self.N, n_features))
        for i, codes in enumerate(self.df['atc_codes']):
            for code in codes:
                c_str = str(code)
                if c_str in vocab_index:
                    tf_matrix[i, vocab_index[c_str]] = 1.0
        print("[2] Ma trận TF: ")
        print(tf_matrix[0:10])
        print("*"*30)

        # 3. Tính IDF (Inverse Document Frequency) 
        # IDF(t, D) = log(N / nt)
        df_counts = np.sum(tf_matrix, axis=0)
        df_counts[df_counts == 0] = 1  # Tránh lỗi chia cho 0
        idf_vector = np.log(self.N / df_counts)
        print(f"[3] IDF Vector (14 giá trị):")
        print(idf_vector)
        print("-"*30)

        # 4. Tính TF-IDF và Chuẩn hóa L2 (L2 Normalization)
        # Chuẩn hóa L2 giúp việc tính tích vô hướng tương đương với Cosine Similarity 
        tfidf_matrix = tf_matrix * idf_vector
        norms = np.sqrt(np.sum(tfidf_matrix**2, axis=1, keepdims=True))
        norms[norms == 0] = 1.0 
        tfidf_norm = tfidf_matrix / norms

        #Ktra giá trị
        print("[4]: TF-IDF")
        for i in range(10):
            print(f" {self.drug_ids[i]}: {tfidf_norm[i]}")
        print("-"*30)
        
        # 5. Tính Cosine Similarity (Dot Product)
        sim_matrix = np.dot(tfidf_norm, tfidf_norm.T)
        print(f"[5] Kích thước Ma trận Tương đồng: {sim_matrix.shape}")
        
        # 6. Chuyển thành DataFrame và lưu kết quả
        df_sim = pd.DataFrame(sim_matrix, index=self.drug_ids, columns=self.drug_ids)
        df_sim.to_csv(output_path)
        
        print(f" Đã hoàn thành và lưu ma trận tương đồng ATC tại: {output_path}")
        return df_sim

INPUT_FILE = '/kaggle/working/drug_node_final.csv' 
atc_gen = ATCFeatureGenerator(INPUT_FILE)
df_atc_sim = atc_gen.compute_atc_similarity('feature_ATC.csv')

Bắt đầu tính toán đặc trưng ATC cho 1608 loại thuốc...
  -> Tìm thấy 14 mã ATC Level 1 duy nhất.
[1] Vocabulary (14 cụm): ['A', 'B', 'C', 'D', 'G', 'H', 'J', 'L', 'M', 'N', 'P', 'R', 'S', 'V']
******************************
[2] Ma trận TF: 
[[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]
******************************
[3] IDF Vector (14 giá trị):
[2.10974689 3.33969518 1.92742533 2.57056209 2.33932133 3.91701055
 1.95779643 2.25284773 2.85014696 1.57761148 3.59855682 2.515212
 2.53855936 3.37541326]
------------------------------
[4]: TF-IDF
 DB00006: [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.

In [25]:
atc = pd.read_csv('/kaggle/working/feature_ATC.csv', index_col = 0)
atc

,DB00006,DB00014,DB00035,DB00080,DB00091,DB00104,DB00115,DB00122,DB00126,DB00130,...,DB12958,DB12967,DB12975,DB13025,DB13028,DB13069,DB13132,DB13136,DB13142,DB13143
DB00006,1.0,0.000000,0.0,0.0,0.000000,0.0,1.0,0.0,0.000000,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,1.0,0.0,0.0
DB00014,0.0,1.000000,0.0,0.0,0.663763,0.0,0.0,0.0,0.000000,0.0,...,0.0,1.000000,0.0,0.0,0.0,1.000000,0.0,0.0,0.0,0.0
DB00035,0.0,0.000000,1.0,0.0,0.000000,1.0,0.0,0.0,0.000000,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
DB00080,0.0,0.000000,0.0,1.0,0.000000,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.000000,0.0,0.0,1.0,0.000000,0.0,0.0,0.0,0.0
DB00091,0.0,0.663763,0.0,0.0,1.000000,0.0,0.0,0.0,0.469312,0.0,...,0.0,0.663763,0.0,0.0,0.0,0.663763,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DB13069,0.0,1.000000,0.0,0.0,0.663763,0.0,0.0,0.0,0.000000,0.0,...,0.0,1.000000,0.0,0.0,0.0,1.000000,0.0,0.0,0.0,0.0
DB13132,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.000000,1.0,0.0,0.0,0.000000,1.0,0.0,0.0,0.0
DB13136,1.0,0.000000,0.0,0.0,0.000000,0.0,1.0,0.0,0.000000,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,1.0,0.0,0.0
DB13142,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.521478,1.0,...,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,1.0,0.0


---

# 2. MeSH

In [26]:
import pandas as pd
import numpy as np
import ast
import os

In [35]:
class MeSHFeatureGenerator:
    """
    TÍNH TOÁN ĐẶC TRƯNG MESH (Drug MeSH-based similarity)
    - Input: File CSV chứa cột 'mesh_terms' dạng list.
    - Output: Ma trận tương đồng Cosine Similarity.
    """
    def __init__(self, input_path):
        if not os.path.exists(input_path):
            raise FileNotFoundError(f"Không tìm thấy file {input_path}")
        
        self.df = pd.read_csv(input_path)
        self.drug_ids = self.df['drugbank_id'].tolist()
        self.N = len(self.drug_ids)
        
        # Chuyển chuỗi văn bản thành danh sách Python thực tế
        target_col = 'mesh_terms' if 'mesh_terms' in self.df.columns else 'mesh'
        self.df[target_col] = self.df[target_col].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else []
        )
        self.target_col = target_col

    def compute_mesh_similarity(self, output_path='feature_MESH.csv'):
        np.set_printoptions(threshold=10, precision=4, suppress=True)
        print(f"Bắt đầu tính toán đặc trưng MeSH cho {self.N} loại thuốc...")
        
        # 1. Tạo bộ từ vựng MeSH (Vocabulary)
        unique_mesh = set()
        for terms in self.df[self.target_col]:
            unique_mesh.update({str(t) for t in terms if str(t)})
        
        vocab = sorted(list(unique_mesh))
        vocab_index = {term: i for i, term in enumerate(vocab)}
        n_features = len(vocab)
        print(f"  -> Tìm thấy {n_features} thuật ngữ MeSH duy nhất.")
        print(f"[1] Vocabulary sample: {vocab[:20]}...") 
        print("-"*50)

        # 2. Xây dựng ma trận TF (Binary Term Frequency)
        tf_matrix = np.zeros((self.N, n_features))
        for i, terms in enumerate(self.df[self.target_col]):
            for term in terms:
                t_str = str(term)
                if t_str in vocab_index:
                    tf_matrix[i, vocab_index[t_str]] = 1.0
        print("[2] Ma trận TF: ")
        with np.printoptions(threshold=np.inf): 
            print(tf_matrix[0]) 
            print(f"\n=> Tổng số mã 1.0 tìm thấy: {np.sum(tf_matrix)}")
        print("-"*50)

        # 3. Tính IDF (Inverse Document Frequency)
        df_counts = np.sum(tf_matrix, axis=0)
        df_counts[df_counts == 0] = 1 
        idf_vector = np.log(self.N / df_counts)
        print(f"[3] IDF Vector :")
        print(f"{idf_vector[:5]}")
        print("-"*30)

        # 4. Tính TF-IDF và Chuẩn hóa L2
        tfidf_matrix = tf_matrix * idf_vector
        norms = np.sqrt(np.sum(tfidf_matrix**2, axis=1, keepdims=True))
        norms[norms == 0] = 1.0 
        tfidf_norm = tfidf_matrix / norms

        print("[4]: TF-IDF (Sau chuẩn hóa L2 - 10 thuốc đầu)")
        for i in range(10):
            # Chỉ in các giá trị khác 0 để dễ kiểm tra vì MeSH rất thưa (sparse)
            non_zero_indices = np.nonzero(tfidf_norm[i])[0]
            vals = tfidf_norm[i][non_zero_indices]
            print(f" {self.drug_ids[i]}: {vals}")
        print("-"*50)
        
        # 5. Tính Cosine Similarity (Dot Product)
        sim_matrix = np.dot(tfidf_norm, tfidf_norm.T)
        print(f"[5] Kích thước Ma trận Tương đồng: {sim_matrix.shape}")
        
        # 6. Chuyển thành DataFrame và lưu kết quả
        df_sim = pd.DataFrame(sim_matrix, index=self.drug_ids, columns=self.drug_ids)
        df_sim.to_csv(output_path)
        
        print(f" Đã hoàn thành và lưu ma trận tương đồng MeSH tại: {output_path}")
        return df_sim

INPUT_FILE = '/kaggle/working/drug_node_final.csv' 
mesh_gen = MeSHFeatureGenerator(INPUT_FILE)
df_mesh_sim = mesh_gen.compute_mesh_similarity('feature_MESH.csv')

Bắt đầu tính toán đặc trưng MeSH cho 1608 loại thuốc...
  -> Tìm thấy 1365 thuật ngữ MeSH duy nhất.
[1] Vocabulary sample: ['D000019', 'D000020', 'D000021', 'D000066888', 'D000067856', 'D000068760', 'D000068776', 'D000079', 'D000081', 'D000083', 'D000085', 'D000143', 'D000144', 'D000145', 'D000146', 'D000147', 'D000148', 'D000166', 'D000172', 'D000212']...
--------------------------------------------------
[2] Ma trận TF: 
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0

In [36]:
mesh = pd.read_csv('/kaggle/working/feature_MESH.csv', index_col = 0)
mesh

,DB00006,DB00014,DB00035,DB00080,DB00091,DB00104,DB00115,DB00122,DB00126,DB00130,...,DB12958,DB12967,DB12975,DB13025,DB13028,DB13069,DB13132,DB13136,DB13142,DB13143
DB00006,1.000000,0.153103,0.150545,0.124608,0.097074,0.123710,0.000059,0.001424,0.001818,0.040784,...,0.000000,0.025181,0.000680,0.002546,0.000422,0.000459,0.000255,0.249611,0.000000,0.000000
DB00014,0.153103,1.000000,0.418880,0.125491,0.087352,0.393315,0.001459,0.000220,0.001713,0.041074,...,0.000000,0.033155,0.000685,0.002419,0.000425,0.024193,0.000257,0.000411,0.000000,0.000000
DB00035,0.150545,0.418880,1.000000,0.095337,0.066363,0.094651,0.001109,0.000167,0.001301,0.031204,...,0.000000,0.000481,0.000520,0.001837,0.000323,0.000351,0.000195,0.049266,0.000000,0.000000
DB00080,0.124608,0.125491,0.095337,1.000000,0.364546,0.505170,0.062814,0.000342,0.000108,0.063787,...,0.000000,0.000984,0.038421,0.000611,0.042477,0.000718,0.014406,0.018667,0.000000,0.030721
DB00091,0.097074,0.087352,0.066363,0.364546,1.000000,0.345277,0.044419,0.001522,0.003687,0.043598,...,0.000000,0.026918,0.026260,0.004872,0.029032,0.000491,0.009846,0.012759,0.000000,0.020997
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DB13069,0.000459,0.024193,0.000351,0.000718,0.000491,0.037300,0.000095,0.001840,0.002270,0.000000,...,0.006958,0.057233,0.001093,0.003289,0.003549,1.000000,0.002145,0.003434,0.002628,0.004733
DB13132,0.000255,0.000257,0.000195,0.014406,0.009846,0.000396,0.000053,0.013002,0.001261,0.000000,...,0.033307,0.002939,0.265173,0.011988,0.038326,0.002145,1.000000,0.012516,0.001460,0.020705
DB13136,0.249611,0.000411,0.049266,0.018667,0.012759,0.018532,0.011002,0.036940,0.002018,0.000000,...,0.006186,0.004706,0.000972,0.066047,0.057055,0.003434,0.012516,1.000000,0.002337,0.250894
DB13142,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.001252,0.043411,0.000000,...,0.005853,0.003602,0.000000,0.002238,0.002415,0.002628,0.001460,0.002337,1.000000,0.003982


---

# 3. ADE

In [37]:
import pandas as pd
import numpy as np
import ast
import os

In [38]:
class ADEFeatureGenerator:
    def __init__(self, input_path, col_name='side_effect_codes'):
        if not os.path.exists(input_path):
            raise FileNotFoundError(f"Không tìm thấy file {input_path}")
        
        self.df = pd.read_csv(input_path)    
        self.drug_ids = self.df['drugbank_id'].tolist()
        self.N = len(self.drug_ids)
        self.col_name = col_name
        self.df[self.col_name] = self.df[self.col_name].apply(self._simple_clean)

    def _simple_clean(self, item):
        """
        Chiến thuật: Xóa hết ngoặc [], xóa hết dấu nháy '" và cắt theo dấu phẩy.
        """
        if isinstance(item, list):
            return [str(x) for x in item]
            
        if isinstance(item, str):
            try:
                val = ast.literal_eval(item)
                if isinstance(val, list):
                    return [str(x) for x in val]
            except:
                pass 

            clean_str = item.replace('[', '').replace(']', '').replace("'", "").replace('"', "")
        
            if clean_str.strip(): # Nếu chuỗi không rỗng
                return [x.strip() for x in clean_str.split(',')]
                
        return []

    def compute_ade_similarity(self, output_path='feature_ADE.csv'):
        print(f"Bắt đầu tính toán cho {self.N} thuốc...")
        
        # 1. Tạo bộ từ vựng (Vocabulary)
        unique_ade = set()
        for codes in self.df[self.col_name]:
            unique_ade.update({str(c) for c in codes if str(c)})
        
        vocab = sorted(list(unique_ade))
        vocab_index = {code: i for i, code in enumerate(vocab)}
        n_features = len(vocab)
        
        print(f" -> Số lượng mã tác dụng phụ: {n_features}")
        print("-"*50)

        # 2. Tạo Ma trận TF (0 hoặc 1)
        tf_matrix = np.zeros((self.N, n_features))
        for i, codes in enumerate(self.df[self.col_name]):
            for code in codes:
                c_str = str(code)
                if c_str in vocab_index:
                    tf_matrix[i, vocab_index[c_str]] = 1.0
                    
        # Kiểm tra thuốc đầu tiên có bao nhiêu mã được đánh dấu 1.0
        indices = np.where(tf_matrix[0] == 1.0)[0]
        print(f" -> Kiểm tra thuốc {self.drug_ids[0]}:")
        print(f"    - Số lượng mã tìm thấy trong Vocab: {len(indices)}")
        if len(indices) > 0:
            print(f"    - Các mã tại vị trí cột: {indices}")
            print(f"    - Tương ứng mã UMLS: {[vocab[i] for i in indices[:5]]}...")
        else:
            print("Không tìm thấy mã nào (Vector toàn số 0).")

        # 3. Tính IDF (Logarit)
        df_counts = np.sum(tf_matrix, axis=0)
        df_counts[df_counts == 0] = 1 
        idf_vector = np.log(self.N / df_counts)
       
        print(f" -> Các giá trị : {idf_vector[:20]}")
        print(f" -> Giá trị IDF cao nhất (Hiếm nhất): {np.max(idf_vector):.4f}")
        print("-"*50)
        
        # 4. Tính TF-IDF & Chuẩn hóa L2
        tfidf_matrix = tf_matrix * idf_vector
        norms = np.sqrt(np.sum(tfidf_matrix**2, axis=1, keepdims=True))
        norms[norms == 0] = 1.0 
        tfidf_norm = tfidf_matrix / norms

        print(f" Ma trận TF-IDF sau khi chuẩn hóa L2:")
        print(f" -> Kiểm tra thuốc {self.drug_ids[0]}:")
        non_zero_vals = tfidf_norm[0][indices]
        if len(non_zero_vals) > 0:
            print(f"    - Các giá trị trọng số : {non_zero_vals[:20]}...")
        else:
            print("    - Vector rỗng.")
        # Kiểm tra độ dài vector (phải = 1.0)
        norm_val = np.linalg.norm(tfidf_norm[0])
        print(f"    - Tổng bình phương (Norm Check): {norm_val:.4f}")
        
        # 5. Tính Cosine Similarity
        sim_matrix = np.dot(tfidf_norm, tfidf_norm.T)
        print(f" -> Kích thước ma trận kết quả: {sim_matrix.shape}")
        
        # 6. Lưu kết quả 
        df_sim = pd.DataFrame(sim_matrix, index=self.drug_ids, columns=self.drug_ids)
        df_sim.to_csv(output_path)
        print(f"Đã xong! File lưu tại: {output_path}")
        
        return df_sim


NODE_FILE_PATH = '/kaggle/working/drug_node_final.csv'
OUTPUT_FILE = 'feature_ADE.csv'

ade_gen = ADEFeatureGenerator(NODE_FILE_PATH, col_name='side_effect_codes')
df_ade = ade_gen.compute_ade_similarity(OUTPUT_FILE)

Bắt đầu tính toán cho 1608 thuốc...
 -> Số lượng mã tác dụng phụ: 5376
--------------------------------------------------
 -> Kiểm tra thuốc DB00006:
    - Số lượng mã tìm thấy trong Vocab: 119
    - Các mã tại vị trí cột: [   4   56   57 ... 5174 5321 5375]
    - Tương ứng mã UMLS: ['C0000737', 'C0002792', 'C0002871', 'C0002962', 'C0003467']...
 -> Các giá trị : [5.9965 3.3574 2.1898 ... 7.3827 7.3827 3.6215]
 -> Giá trị IDF cao nhất (Hiếm nhất): 7.3827
--------------------------------------------------
 Ma trận TF-IDF sau khi chuẩn hóa L2:
 -> Kiểm tra thuốc DB00006:
    - Các giá trị trọng số : [0.0279 0.0361 0.0372 ... 0.0691 0.1327 0.021 ]...
    - Tổng bình phương (Norm Check): 1.0000
 -> Kích thước ma trận kết quả: (1608, 1608)
Đã xong! File lưu tại: feature_ADE.csv


In [39]:
ade = pd.read_csv('/kaggle/working/feature_ADE.csv', index_col = 0)
ade

,DB00006,DB00014,DB00035,DB00080,DB00091,DB00104,DB00115,DB00122,DB00126,DB00130,...,DB12958,DB12967,DB12975,DB13025,DB13028,DB13069,DB13132,DB13136,DB13142,DB13143
DB00006,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.059254,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
DB00014,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
DB00035,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
DB00080,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
DB00091,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DB13069,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
DB13132,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
DB13136,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
DB13142,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


---

# 4. SMILES

In [40]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 47.2 MB/s eta 0:00:00:00:0100:01


In [41]:
import pandas as pd
import numpy as np
import os
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

In [42]:
class ChemicalFeatureGenerator:
    def __init__(self, input_path, smiles_col='smiles'):
        if not os.path.exists(input_path):
            raise FileNotFoundError(f"Không tìm thấy file {input_path}")
        
        self.df = pd.read_csv(input_path)
        if 'drugbank_id' not in self.df.columns and 'id' in self.df.columns:
            self.df.rename(columns={'id': 'drugbank_id'}, inplace=True)
            
        self.drug_ids = self.df['drugbank_id'].tolist()
        self.N = len(self.drug_ids)
        self.smiles_col = smiles_col
    
    def compute_chemical_similarity(self, output_path='feature_Chemical.csv'):
        print(f"Bắt đầu xử lý cho {self.N} thuốc...")
        
        mols = []
        for sm in self.df[self.smiles_col]:
            if isinstance(sm, str) and len(sm) > 0:
                # Hàm này biến chuỗi ký tự thành đối tượng đồ thị hóa học
                mol = Chem.MolFromSmiles(sm) 
                mols.append(mol)
            else:
                mols.append(None)
        
        print(f"\n[BƯỚC 1] Chuyển đổi SMILES -> Mol Object:")
        print(f" -> Dữ liệu gốc (Thuốc đầu): {self.df[self.smiles_col].iloc[0][:50]}... ")
        print(f" -> Dữ liệu mới (Thuốc đầu): {mols[0]} (Dạng Object RDKit)")
        if mols[0]:
            print(f" -> Kiểm tra: Thuốc này có {mols[0].GetNumAtoms()} nguyên tử.")

       
        # BƯỚC 2: Mol (Object) -> Fingerprint (Bit Vector)
        # Theo bài báo: Morgan (Circular) Fingerprint, bán kính 2, 1024 bit
        fps = []
        for mol in mols:
            if mol is not None:
                # Hàm này "băm" cấu trúc hóa học thành dãy bit 0/1
                fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)
                fps.append(fp)
            else:
                fps.append(None)

        print(f"\n[BƯỚC 2] Chuyển đổi Mol -> Fingerprint (Bit Vector):")
        # Ta xem các vị trí bit được bật lên (có giá trị 1)
        if fps[0]:
            on_bits = list(fps[0].GetOnBits())
            print(f" -> Dữ liệu dạng Vector nhị phân (1024 chiều).")
            print(f" -> Các vị trí bit '1' của thuốc đầu tiên: {on_bits[:10]}... (Tổng {len(on_bits)} bit bật)")
            # Mỗi số là một đặc điểm cấu trúc hóa học con (substructure) có mặt trong thuốc.

        # BƯỚC 3: Tính Ma trận Tương đồng (Tanimoto Similarity)
        # Ma trận NxN chứa điểm số từ 0.0 đến 1.0
        sim_matrix = np.zeros((self.N, self.N))
        
        print(f"\n[BƯỚC 3] Tính độ tương đồng Tanimoto (NxN)...")
        for i in range(self.N):
            for j in range(i, self.N): # Chỉ tính tam giác trên
                if fps[i] is not None and fps[j] is not None:
                    # Công thức: (Bit chung) / (Tổng bit của cả 2 - Bit chung)
                    sim = DataStructs.TanimotoSimilarity(fps[i], fps[j])
                    sim_matrix[i, j] = sim
                    sim_matrix[j, i] = sim # Đối xứng
                else:
                    sim_matrix[i, j] = 0.0
                    sim_matrix[j, i] = 0.0

        print(f" -> Hoàn thành. Kích thước ma trận: {sim_matrix.shape}")
        print(f" -> Mẫu kết quả (DB00006 vs 10 thuốc đầu):")
        print(f"    {sim_matrix[0, :10]}")

 
        # BƯỚC 4: Lưu file
        df_sim = pd.DataFrame(sim_matrix, index=self.drug_ids, columns=self.drug_ids)
        df_sim.to_csv(output_path)
        
        print(f"\n[HOÀN TẤT] Đã lưu file tại: {output_path}")
        return df_sim

NODE_FILE_PATH = '/kaggle/working/drug_node_final.csv'
OUTPUT_FILE = 'feature_SMILES.csv'

chem_gen = ChemicalFeatureGenerator(NODE_FILE_PATH, smiles_col='smiles')
df_chem = chem_gen.compute_chemical_similarity(OUTPUT_FILE)

Bắt đầu xử lý cho 1608 thuốc...

[BƯỚC 1] Chuyển đổi SMILES -> Mol Object:
 -> Dữ liệu gốc (Thuốc đầu): CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@H](C... 
 -> Dữ liệu mới (Thuốc đầu): <rdkit.Chem.rdchem.Mol object at 0x7abd16257840> (Dạng Object RDKit)
 -> Kiểm tra: Thuốc này có 155 nguyên tử.

[BƯỚC 2] Chuyển đổi Mol -> Fingerprint (Bit Vector):
 -> Dữ liệu dạng Vector nhị phân (1024 chiều).
 -> Các vị trí bit '1' của thuốc đầu tiên: [1, 4, 33, 41, 42, 64, 79, 80, 83, 86]... (Tổng 103 bit bật)

[BƯỚC 3] Tính độ tương đồng Tanimoto (NxN)...
 -> Hoàn thành. Kích thước ma trận: (1608, 1608)
 -> Mẫu kết quả (DB00006 vs 10 thuốc đầu):
    [1.     0.3711 0.4071 0.226  0.1154 0.2013 0.1139 0.0268 0.0667 0.1296]

[HOÀN TẤT] Đã lưu file tại: feature_SMILES.csv


In [43]:
smiles = pd.read_csv('/kaggle/working/feature_SMILES.csv', index_col=0)
smiles

,DB00006,DB00014,DB00035,DB00080,DB00091,DB00104,DB00115,DB00122,DB00126,DB00130,...,DB12958,DB12967,DB12975,DB13025,DB13028,DB13069,DB13132,DB13136,DB13142,DB13143
DB00006,1.000000,0.371069,0.407143,0.225989,0.115385,0.201258,0.113861,0.026786,0.066667,0.129630,...,0.110236,0.076923,0.119718,0.106870,0.105634,0.081481,0.052239,0.076923,0.096296,0.148760
DB00014,0.371069,1.000000,0.306250,0.244565,0.120482,0.260870,0.133971,0.040984,0.085271,0.063492,...,0.100719,0.061538,0.132450,0.105634,0.119205,0.097222,0.085106,0.078125,0.073826,0.110294
DB00035,0.407143,0.306250,1.000000,0.202312,0.130137,0.281690,0.125000,0.019231,0.091743,0.130000,...,0.100000,0.063636,0.102941,0.070866,0.104478,0.104839,0.056000,0.114286,0.069231,0.111111
DB00080,0.225989,0.244565,0.202312,1.000000,0.178344,0.346667,0.145631,0.041322,0.120968,0.090164,...,0.085714,0.053846,0.089744,0.090909,0.120000,0.082759,0.109489,0.087302,0.111888,0.119403
DB00091,0.115385,0.120482,0.130137,0.178344,1.000000,0.152174,0.128655,0.037500,0.129412,0.071429,...,0.068627,0.056180,0.076271,0.086538,0.157407,0.065421,0.112245,0.068182,0.104762,0.091837
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DB13069,0.081481,0.097222,0.104839,0.082759,0.065421,0.091667,0.057692,0.078431,0.062500,0.068966,...,0.125000,0.081967,0.137931,0.148649,0.102273,1.000000,0.065789,0.064516,0.047619,0.097222
DB13132,0.052239,0.085106,0.056000,0.109489,0.112245,0.076923,0.095890,0.020408,0.105263,0.036364,...,0.027027,0.033898,0.056180,0.038961,0.095238,0.065789,1.000000,0.089286,0.064103,0.057143
DB13136,0.076923,0.078125,0.114286,0.087302,0.068182,0.110000,0.058394,0.000000,0.116279,0.050000,...,0.089286,0.022222,0.053333,0.065574,0.069444,0.064516,0.089286,1.000000,0.079365,0.072727
DB13142,0.096296,0.073826,0.069231,0.111888,0.104762,0.117647,0.120805,0.055556,0.166667,0.142857,...,0.024691,0.046154,0.074468,0.087500,0.100000,0.047619,0.064103,0.079365,1.000000,0.140845


---

# 5. Xây dựng đồ thị

In [44]:
import pandas as pd
import networkx as nx
import os
import pickle

In [45]:
class DDIGraphBuilder:
    """
    Class quản lý quy trình xây dựng, phân tích và lưu trữ đồ thị tương tác thuốc (DDI).
    """
    
    def __init__(self, edges_file, nodes_file):
        """
        Khởi tạo với đường dẫn file cạnh và file nút.
        """
        self.edges_file = edges_file
        self.nodes_file = nodes_file
        self.G = None
        self.df_edges = None
        self.df_nodes = None

    def build_graph(self):
        """
        Bước 1: Đọc dữ liệu và xây dựng đồ thị NetworkX.
        """
        print("BƯỚC 1: BẮT ĐẦU XÂY DỰNG ĐỒ THỊ DDI")

        # 1. Đọc dữ liệu
        try:
            self.df_edges = pd.read_csv(self.edges_file)
            self.df_nodes = pd.read_csv(self.nodes_file)
            print(f"-> Đã đọc file edges: {len(self.df_edges)} dòng.")
            print(f"-> Đã đọc file nodes: {len(self.df_nodes)} dòng.")
        except FileNotFoundError as e:
            print(f"[LỖI] Không tìm thấy file: {e}")
            return None

        # 2. Tạo đồ thị từ danh sách cạnh
        # create_using=nx.Graph() đảm bảo là đồ thị vô hướng
        self.G = nx.from_pandas_edgelist(
            self.df_edges, 
            source='source', 
            target='target', 
            create_using=nx.Graph()
        )

        # 3. Gắn thuộc tính cho node
        if 'drugbank_id' in self.df_nodes.columns:
            # Chuyển đổi DataFrame thành dictionary với key là drugbank_id
            node_attr_dict = self.df_nodes.set_index('drugbank_id').to_dict('index')
            nx.set_node_attributes(self.G, node_attr_dict)
            print("-> Đã gắn thuộc tính (attributes) cho các node thành công.")
        else:
            print("[CẢNH BÁO] Không tìm thấy cột 'drugbank_id'. Các node sẽ không có thuộc tính!")

        return self.G

    def analyze_graph(self):
        """
        Bước 2: Phân tích các chỉ số cơ bản của đồ thị.
        """
        if self.G is None:
            print("[LỖI] Đồ thị chưa được khởi tạo. Hãy chạy build_graph() trước.")
            return

        print("\n" + "="*40)
        print("BƯỚC 2: KẾT QUẢ THỐNG KÊ ĐỒ THỊ")
        print("="*40)
        
        n_nodes = self.G.number_of_nodes()
        n_edges = self.G.number_of_edges()
        density = nx.density(self.G)
        isolates = list(nx.isolates(self.G))

        print(f"• Số lượng thuốc (Nodes):        {n_nodes}")
        print(f"• Số lượng tương tác (Edges):    {n_edges}")
        print(f"• Mật độ đồ thị (Density):       {density:.6f}")

        # Kiểm tra node cô lập
        if isolates:
            print(f"• [CẢNH BÁO] Có {len(isolates)} thuốc bị cô lập (không có tương tác).")
        else:
            print("• Không có thuốc bị cô lập.")

        # Kiểm tra tính toàn vẹn dữ liệu (Node trong Edges vs Node trong Attributes)
        nodes_in_edges = set(self.G.nodes())
        nodes_in_attrs = set(self.df_nodes['drugbank_id']) if self.df_nodes is not None and 'drugbank_id' in self.df_nodes.columns else set()
        
        # Node có trong danh sách cạnh nhưng thiếu thông tin trong file nodes
        missing_info = nodes_in_edges - nodes_in_attrs
    
    def inspect_node(self, node_id):
        """
        Bước 3: Kiểm tra chi tiết một node cụ thể và các láng giềng.
        """
        if self.G is None:
            return

        print("\n" + "="*40)
        print(f"BƯỚC 3: KIỂM TRA CHI TIẾT NODE: {node_id}")
        print("="*40)

        if node_id not in self.G:
            print(f"[LỖI] Node ID '{node_id}' không tồn tại trong đồ thị.")
            return

        # Lấy thuộc tính của node cha
        node_attrs = self.G.nodes[node_id]
        print(f"-> Thuộc tính node cha: {node_attrs}")

        # Lấy danh sách láng giềng
        neighbors = list(self.G.neighbors(node_id))
        print(f"-> Tổng số node tương tác (Láng giềng): {len(neighbors)}")

        # In chi tiết 5 láng giềng đầu tiên để kiểm tra
        print("-> Danh sách 5 láng giềng đầu tiên:")
        for i, neighbor in enumerate(neighbors[:5]):
            neighbor_attrs = self.G.nodes[neighbor]
            print(f"   {i+1}. ID: {neighbor} | Attrs: {neighbor_attrs}")
        
        if len(neighbors) > 5:
            print(f"   ... và {len(neighbors) - 5} node khác.")

    def save_graph(self, output_path):
        """
        Bước 4: Lưu đồ thị xuống file .pkl
        """
        if self.G is None:
            return

        print("\n" + "="*40)
        print(f"BƯỚC 4: LƯU ĐỒ THỊ VÀO FILE")
        print("="*40)
        print(f"-> Đang lưu cấu trúc đồ thị vào file: {output_path}...")

        try:
            # Tạo thư mục nếu chưa tồn tại
            os.makedirs(os.path.dirname(output_path), exist_ok=True)
            
            with open(output_path, 'wb') as f:
                pickle.dump(self.G, f)
            
            # Kiểm tra kích thước file
            size_mb = os.path.getsize(output_path) / (1024 * 1024)
            print(f"[THÀNH CÔNG] Đã lưu file! Kích thước: {size_mb:.2f} MB")
        except Exception as e:
            print(f"[LỖI] Không thể lưu file: {e}")


if __name__ == "__main__":
    # 1. Cấu hình đường dẫn
    EDGES_FILE = "/kaggle/working/drug_edge_final.csv"
    NODES_FILE = "/kaggle/working/drug_node_final.csv"
    OUTPUT_FILE = "/kaggle/working/ddi_graph_final.pkl"

    builder = DDIGraphBuilder(EDGES_FILE, NODES_FILE)

    G = builder.build_graph()
    builder.analyze_graph()
    builder.save_graph(OUTPUT_FILE)

BƯỚC 1: BẮT ĐẦU XÂY DỰNG ĐỒ THỊ DDI
-> Đã đọc file edges: 195150 dòng.
-> Đã đọc file nodes: 1608 dòng.
-> Đã gắn thuộc tính (attributes) cho các node thành công.

BƯỚC 2: KẾT QUẢ THỐNG KÊ ĐỒ THỊ
• Số lượng thuốc (Nodes):        1608
• Số lượng tương tác (Edges):    195150
• Mật độ đồ thị (Density):       0.151042
• Không có thuốc bị cô lập.

BƯỚC 4: LƯU ĐỒ THỊ VÀO FILE
-> Đang lưu cấu trúc đồ thị vào file: /kaggle/working/ddi_graph_final.pkl...
[THÀNH CÔNG] Đã lưu file! Kích thước: 4.89 MB


---

# 5. Phân cụm

In [46]:
!pip install python-louvain

import community.community_louvain as community_louvain
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import pickle
import os

In [47]:
class SimpleLouvainPipeline:
    def __init__(self, graph_input):
        # 1. Tải đồ thị
        if isinstance(graph_input, str) and os.path.exists(graph_input):
            print(f"-> Đang đọc file: {graph_input}")
            with open(graph_input, 'rb') as f:
                self.G = pickle.load(f)
        elif isinstance(graph_input, nx.Graph):
            self.G = graph_input
        else:
            raise ValueError("Input không hợp lệ!")
        
        self.partition = None

    def run_and_save(self, csv_path='drug_communities.csv', pkl_path='ddi_graph_with_comm.pkl'):
        # 2. Chạy thuật toán Louvain
        print("-> Đang phân cụm...")
        # random_state=42 giúp kết quả cố định mỗi lần chạy
        self.partition = community_louvain.best_partition(self.G, random_state=42)
        
        # 3. Tính điểm chất lượng (Modularity)
        q_score = community_louvain.modularity(self.partition, self.G)
        num_comm = len(set(self.partition.values()))
        print(f"-> Kết quả: {num_comm} cộng đồng. Modularity (Q) = {q_score:.4f}")

        # 4. Gán nhãn cộng đồng vào đồ thị
        nx.set_node_attributes(self.G, self.partition, 'community')

        # 5. Lưu CSV danh sách cộng đồng
        df = pd.DataFrame.from_dict(self.partition, orient='index', columns=['community_id'])
        df.index.name = 'drugbank_id'
        df.to_csv(csv_path) # Pandas tự động lưu index, không cần reset_index rườm rà
        
        # 6. Lưu file đồ thị .pkl mới
        with open(pkl_path, 'wb') as f:
            pickle.dump(self.G, f)
            
        print(f"-> Đã lưu xong: '{csv_path}' và '{pkl_path}'")
        return df

INPUT_PKL = "/kaggle/working/ddi_graph_final.pkl"

pipeline = SimpleLouvainPipeline(INPUT_PKL)
df_result = pipeline.run_and_save()

-> Đang đọc file: /kaggle/working/ddi_graph_final.pkl
-> Đang phân cụm...
-> Kết quả: 5 cộng đồng. Modularity (Q) = 0.2866
-> Đã lưu xong: 'drug_communities.csv' và 'ddi_graph_with_comm.pkl'


In [48]:
import pickle
import networkx as nx

# Đường dẫn file
FILE_PATH = '/kaggle/working/ddi_graph_with_comm.pkl'

# 1. Đọc file
with open(FILE_PATH, 'rb') as f:
    G = pickle.load(f)
    
sample_nodes = list(G.nodes(data=True))[:3]

for node_id, attributes in sample_nodes:
    print(f"ID Thuốc: {node_id}")
    print(f"Thông tin: {attributes}")
    print("-" * 30)

ID Thuốc: DB00006
Thông tin: {'Unnamed: 0': 0, 'name': 'Bivalirudin', 'smiles': 'CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@H](CCC(O)=O)NC(=O)[C@H](CC1=CC=CC=C1)NC(=O)[C@H](CC(O)=O)NC(=O)CNC(=O)[C@H](CC(N)=O)NC(=O)CNC(=O)CNC(=O)CNC(=O)CNC(=O)[C@@H]1CCCN1C(=O)[C@H](CCCNC(N)=N)NC(=O)[C@@H]1CCCN1C(=O)[C@H](N)CC1=CC=CC=C1)C(=O)N1CCC[C@H]1C(=O)N[C@@H](CCC(O)=O)C(=O)N[C@@H](CCC(O)=O)C(=O)N[C@@H](CC1=CC=C(O)C=C1)C(=O)N[C@@H](CC(C)C)C(O)=O', 'atc_codes': "['B']", 'mesh_terms': "['D000602', 'D000925', 'D058833', 'D000991', 'D020164', 'D004791', 'D005343', 'D006401', 'D045504', 'D010455', 'D020228', 'D011480', 'D011506', 'D015842', 'D015843', 'D045506']", 'side_effect_codes': "['C0035126', 'C0002962', 'C0042075', 'C0917801', 'C0020517', 'C0687713', 'C0518015', 'C1696110', 'C0917798', 'C0031256', 'C0341512', 'C0919980', 'C0015469', 'C0576995', 'C0042510', 'C0018944', 'C0032326', 'C0019064', 'C0009492', 'C0010072', 'C0014591', 'C0042963', 'C0009450', 'C0042514', 'C0151828', 'C0151705', 'C015173

---

# 6. Processing 8 feature Topo

In [49]:
import networkx as nx
import pandas as pd
import numpy as np
import itertools
import os
import pickle

In [50]:
class LinkPredictionPipeline:
    def __init__(self, graph_path):
        # 1. Tải đồ thị
        if not os.path.exists(graph_path):
            raise FileNotFoundError(f"Không tìm thấy file: {graph_path}")
            
        print(f"-> Đang tải đồ thị từ: {graph_path}...")
        with open(graph_path, 'rb') as f:
            self.G = pickle.load(f)
        
        self.df_features = None
        
        first_node = list(self.G.nodes())[0]
        if 'community' not in self.G.nodes[first_node]:
            raise ValueError(
                "[LỖI] Đồ thị này chưa có nhãn 'community'! "
            )
        
        print("-> Đồ thị hợp lệ (đã có nhãn Community). Sẵn sàng tính toán.")

    def compute_8_features(self):
        print("\n" + "="*40)
        print("BƯỚC 1: TÍNH TOÁN 8 ĐẶC TRƯNG TOPO")
        print("="*40)

        # 1. Tạo danh sách tất cả cặp nút (u, v)
        print("-> Đang sinh tất cả các cặp nút (Nodes Pairs)...")
        nodes = list(self.G.nodes())
        all_pairs = list(itertools.combinations(nodes, 2))
        print(f"-> Tổng số cặp cần tính: {len(all_pairs)}")

        # 2. Khởi tạo DataFrame
        self.df_features = pd.DataFrame(all_pairs, columns=['u', 'v'])

        # --- NHÓM 1: ĐẶC TRƯNG LÂN CẬN (NEIGHBOR-BASED) ---
        print("-> Đang tính nhóm đặc trưng lân cận (CN, RAI, JC, AAI, PA)...")
        
        # Common Neighbors (CN)
        self.df_features['cn'] = self.df_features.apply(
            lambda x: len(list(nx.common_neighbors(self.G, x['u'], x['v']))), axis=1
        )

        self.df_features['rai'] = [p[2] for p in nx.resource_allocation_index(self.G, all_pairs)]
        self.df_features['jc'] = [p[2] for p in nx.jaccard_coefficient(self.G, all_pairs)]
        self.df_features['aai'] = [p[2] for p in nx.adamic_adar_index(self.G, all_pairs)]
        self.df_features['pa'] = [p[2] for p in nx.preferential_attachment(self.G, all_pairs)]

        # --- NHÓM 2: ĐẶC TRƯNG CỘNG ĐỒNG (COMMUNITY-BASED) ---
        print("-> Đang tính nhóm đặc trưng cộng đồng (CCN, CRA, WIC)...")
        self.df_features['ccn'] = [p[2] for p in nx.cn_soundarajan_hopcroft(self.G, all_pairs, community='community')]
        self.df_features['cra'] = [p[2] for p in nx.ra_index_soundarajan_hopcroft(self.G, all_pairs, community='community')]
        self.df_features['wic'] = [p[2] for p in nx.within_inter_cluster(self.G, all_pairs, community='community')]

        # --- NHÓM 3: NHÃN (GROUND TRUTH) ---
        print("-> Đang gán nhãn (Labeling)...")
        true_edges = set(frozenset(e) for e in self.G.edges())
        
        self.df_features['label'] = self.df_features.apply(
            lambda x: 1 if frozenset((x['u'], x['v'])) in true_edges else 0, axis=1
        )

        print("-> Hoàn tất tính toán.")
        return self.df_features

    def save_to_csv(self, output_path='8ft_topology.csv'):
        print("\n" + "="*40)
        print("BƯỚC 2: LƯU TRỮ")
        print("="*40)
        
        if self.df_features is not None:
            print(f"-> Đang lưu file tại: {output_path}...")
            self.df_features.to_csv(output_path, index=False)
            print(f"-> Thành công! Kích thước file: {os.path.getsize(output_path) / (1024*1024):.2f} MB")
        else:
            print("[Lỗi] Chưa có dữ liệu.")

INPUT_PKL = "/kaggle/working/ddi_graph_with_comm.pkl"  
OUTPUT_CSV = "8ft_topology.csv"


pipeline = LinkPredictionPipeline(INPUT_PKL)
df_result = pipeline.compute_8_features()
pipeline.save_to_csv(OUTPUT_CSV)

-> Đang tải đồ thị từ: /kaggle/working/ddi_graph_with_comm.pkl...
-> Đồ thị hợp lệ (đã có nhãn Community). Sẵn sàng tính toán.

BƯỚC 1: TÍNH TOÁN 8 ĐẶC TRƯNG TOPO
-> Đang sinh tất cả các cặp nút (Nodes Pairs)...
-> Tổng số cặp cần tính: 1292028
-> Đang tính nhóm đặc trưng lân cận (CN, RAI, JC, AAI, PA)...
-> Đang tính nhóm đặc trưng cộng đồng (CCN, CRA, WIC)...
-> Đang gán nhãn (Labeling)...
-> Hoàn tất tính toán.

BƯỚC 2: LƯU TRỮ
-> Đang lưu file tại: 8ft_topology.csv...
-> Thành công! Kích thước file: 114.80 MB


In [51]:
ft_topo = pd.read_csv('/kaggle/working/8ft_topology.csv')
ft_topo.head()

,u,v,cn,rai,jc,aai,pa,ccn,cra,wic,label
0,DB00006,DB01048,8,0.020682,0.033195,1.334637,9338,8,0.000000,0.000000,1
1,DB00006,DB06736,105,0.309514,0.243619,17.860806,67599,193,0.277142,5.176166,1
2,DB00006,DB01418,181,0.944861,0.384289,32.750717,91147,287,0.669466,1.413314,1
3,DB00006,DB00945,157,0.597264,0.264310,27.317535,111244,255,0.449781,1.660989,1
4,DB00006,DB00210,112,0.396567,0.277228,19.566155,63539,203,0.294818,4.333127,1


In [52]:
import pandas as pd
import numpy as np
import os
import gc 

class DataIntegrationPipeline:
    """
    Class chuyên trách ghép nối dữ liệu:
    - Input 1: File Topo (Dạng Edge List - Dài)
    - Input 2: Danh sách các File Sinh học (Dạng Matrix - Rộng)
    - Method: Numpy Vectorized Lookup 
    """
    def __init__(self, topo_path, bio_paths_dict):
        print(f"-> Đang tải file Topo từ: {topo_path}...")
        if not os.path.exists(topo_path):
             raise FileNotFoundError(f"Không tìm thấy file {topo_path}")

        self.df_final = pd.read_csv(topo_path)
        print(f"   Kích thước dữ liệu gốc: {self.df_final.shape}")
        
        self.bio_paths = bio_paths_dict

    def _lookup_matrix_values(self, df_target, matrix, col_name):
        """
        Hàm cốt lõi: Tra cứu giá trị từ Ma trận đưa vào DataFrame bằng Index số học.
        """
        # 1. Tạo từ điển ánh xạ: DrugID -> Số thứ tự (0, 1, 2...)
        # Giả định index của ma trận là DrugID
        drug_to_idx = {drug: i for i, drug in enumerate(matrix.index)}
        
        # 2. Chuyển đổi cột u và v của DataFrame thành số thứ tự (Integer Index)
        # map(): Đổi ID thành số. fillna(-1): Nếu không tìm thấy thì gán -1.
        u_indices = df_target['u'].map(drug_to_idx).fillna(-1).astype(int)
        v_indices = df_target['v'].map(drug_to_idx).fillna(-1).astype(int)
        
        # 3. Tạo mặt nạ lọc (Mask) để chỉ lấy những cặp hợp lệ
        # (Cả u và v đều phải tồn tại trong ma trận sinh học này)
        valid_mask = (u_indices != -1) & (v_indices != -1)
        
        # 4. Khởi tạo mảng kết quả với giá trị mặc định là 0.0
        scores = np.zeros(len(df_target))
        
        # 5. Tra cứu siêu tốc bằng Numpy (Vectorization)
        if valid_mask.any():
            # Chuyển ma trận pandas thành numpy array để truy xuất nhanh
            mat_values = matrix.values
            
            # Cú pháp: mat_values[mảng_hàng, mảng_cột]
            scores[valid_mask] = mat_values[u_indices[valid_mask], v_indices[valid_mask]]
            
        return scores

    def merge_all(self):
        print("\n" + "="*40)
        print("BẮT ĐẦU QUY TRÌNH GHÉP NỐI (MAPPING)")
        print("="*40)
        
        for name, path in self.bio_paths.items():
            print(f"-> Đang xử lý đặc trưng: {name}...")
            
            if not os.path.exists(path):
                print(f"   [CẢNH BÁO] Không tìm thấy file {path}. Bỏ qua.")
                continue
                
            try:
                matrix_df = pd.read_csv(path, index_col=0)
            except Exception as e:
                print(f"   [LỖI] Không đọc được file {path}: {e}")
                continue

            # Thực hiện Lookup
            col_name = f"sim_{name.lower()}" # Ví dụ: sim_atc, sim_mesh
            self.df_final[col_name] = self._lookup_matrix_values(self.df_final, matrix_df, col_name)
            
            print(f"   [Xong] Đã thêm cột '{col_name}'.")
            
            del matrix_df
            gc.collect()

        print("\n-> Cấu trúc dữ liệu cuối cùng:")
        print(self.df_final.info())
        return self.df_final

    def save_dataset(self, output_path='Final_Dataset_DDI.csv'):
        print(f"\n-> Đang lưu file tổng hợp: {output_path}...")
        self.df_final.to_csv(output_path, index=False)
        print("-> [HOÀN TẤT] Đã tạo xong bộ dữ liệu huấn luyện.")
        print(self.df_final.head())

TOPO_FILE = "/kaggle/working/8ft_topology.csv"

BIO_FILES = {
    "ATC": "/kaggle/working/feature_ATC.csv", 
    "MESH": "/kaggle/working/feature_MESH.csv",
    "ADE": "/kaggle/working/feature_ADE.csv",
    "Chemical": "/kaggle/working/feature_SMILES.csv"
}

OUTPUT_FILE = "Final_Dataset_DDI.csv"

try:
    pipeline = DataIntegrationPipeline(TOPO_FILE, BIO_FILES)
    df_final = pipeline.merge_all()
    pipeline.save_dataset(OUTPUT_FILE)
    
except Exception as e:
    print(f"Lỗi chương trình: {e}")

-> Đang tải file Topo từ: /kaggle/working/8ft_topology.csv...
   Kích thước dữ liệu gốc: (1292028, 11)

BẮT ĐẦU QUY TRÌNH GHÉP NỐI (MAPPING)
-> Đang xử lý đặc trưng: ATC...
   [Xong] Đã thêm cột 'sim_atc'.
-> Đang xử lý đặc trưng: MESH...
   [Xong] Đã thêm cột 'sim_mesh'.
-> Đang xử lý đặc trưng: ADE...
   [Xong] Đã thêm cột 'sim_ade'.
-> Đang xử lý đặc trưng: Chemical...
   [Xong] Đã thêm cột 'sim_chemical'.

-> Cấu trúc dữ liệu cuối cùng:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1292028 entries, 0 to 1292027
Data columns (total 15 columns):
 #   Column        Non-Null Count    Dtype  
---  ------        --------------    -----  
 0   u             1292028 non-null  object 
 1   v             1292028 non-null  object 
 2   cn            1292028 non-null  int64  
 3   rai           1292028 non-null  float64
 4   jc            1292028 non-null  float64
 5   aai           1292028 non-null  float64
 6   pa            1292028 non-null  int64  
 7   ccn           1292028 non-null  

In [1]:
import pandas as pd
df = pd.read_csv("/kaggle/input/datasets/vungocthien/dataset-dont-split/Final_Dataset_DDI.csv")
df.tail(40)

,u,v,cn,rai,jc,aai,pa,ccn,cra,wic,label,sim_atc,sim_mesh,sim_ade,sim_chemical
1291988,DB08884,DB08965,0,0.000000,0.000000,0.000000,2,0,0.000000,0.0,0,0.0,0.000000,0.0,0.059701
1291989,DB08884,DB12829,0,0.000000,0.000000,0.000000,2,0,0.000000,0.0,0,0.0,0.001682,0.0,0.114286
1291990,DB08884,DB12667,0,0.000000,0.000000,0.000000,2,0,0.000000,0.0,0,0.0,0.000120,0.0,0.121212
1291991,DB08884,DB12343,0,0.000000,0.000000,0.000000,2,0,0.000000,0.0,0,0.0,0.002377,0.0,0.113636
1291992,DB11135,DB09137,1,0.007634,0.333333,0.205120,4,2,0.007634,1000.0,0,0.0,0.023246,0.0,0.000000
1291993,DB11135,DB06219,0,0.000000,0.000000,0.000000,2,0,0.000000,0.0,0,0.0,0.000072,0.0,0.000000
1291994,DB11135,DB09154,0,0.000000,0.000000,0.000000,2,0,0.000000,0.0,0,0.0,0.030316,0.0,0.000000
1291995,DB11135,DB08621,0,0.000000,0.000000,0.000000,2,0,0.000000,0.0,0,0.0,0.000053,0.0,0.000000
1291996,DB11135,DB08965,0,0.000000,0.000000,0.000000,2,0,0.000000,0.0,0,0.0,0.000000,0.0,0.000000
1291997,DB11135,DB12829,0,0.000000,0.000000,0.000000,2,0,0.000000,0.0,0,0.0,0.008109,0.0,0.000000


---

# 7. Trộn và chia dữ liệu

In [1]:
import pandas as pd
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split # Thêm thư viện chia tập

# 1. Đọc dữ liệu
df = pd.read_csv("/kaggle/input/datasets/vungocthien/dataset-dont-split/Final_Dataset_DDI.csv")

df_shuffled = shuffle(df, random_state=42)
df_shuffled = df_shuffled.reset_index(drop=True)

df_shuffled.to_csv('Dataset_Final.csv', index=False)

df_train, df_test = train_test_split(
    df_shuffled, 
    test_size=0.34, 
    random_state=42, 
    stratify=df_shuffled['label'] 
)

df_train.to_csv('train_data.csv', index=False)
df_test.to_csv('test_data.csv', index=False)

print(f"Kích thước tập Train (66%): {df_train.shape}")
print(f"Kích thước tập Test (34%):  {df_test.shape}")

Kích thước tập Train (66%): (852738, 15)
Kích thước tập Test (34%):  (439290, 15)


In [3]:
import pandas as pd
test = pd.read_csv('/kaggle/working/test_data.csv')
train = pd.read_csv('/kaggle/working/train_data.csv')

In [5]:
test.head(10)

,u,v,cn,rai,jc,aai,pa,ccn,cra,wic,label,sim_atc,sim_mesh,sim_ade,sim_chemical
0,DB00969,DB00355,0,0.000000,0.000000,0.000000,114,0,0.000000,0.000000,0,0.0,0.030621,0.064599,0.094737
1,DB00493,DB01213,0,0.000000,0.000000,0.000000,9,0,0.000000,0.000000,0,0.0,0.015626,0.088198,0.111111
2,DB00927,DB08908,0,0.000000,0.000000,0.000000,822,0,0.000000,0.000000,0,0.0,0.005898,0.000000,0.038462
3,DB00862,DB00851,15,0.022575,0.051020,2.297689,11660,15,0.000000,0.000000,0,0.0,0.162822,0.000000,0.083333
4,DB00080,DB06412,0,0.000000,0.000000,0.000000,850,0,0.000000,0.000000,0,0.0,0.017314,0.000000,0.098592
5,DB09030,DB00748,40,0.060851,0.091954,6.151116,34354,40,0.000000,0.000000,0,0.0,0.030739,0.000000,0.113402
6,DB06701,DB08947,0,0.000000,0.000000,0.000000,119,0,0.000000,0.000000,0,0.0,0.076782,0.000000,0.078125
7,DB01203,DB09488,2,0.006738,0.003521,0.351287,5049,2,0.000000,0.000000,0,0.0,0.025974,0.000000,0.142857
8,DB06777,DB01126,80,0.120901,0.824742,12.303066,7760,159,0.119437,78.921079,0,0.0,0.047946,0.000000,0.175824
9,DB00846,DB00399,78,0.235743,0.330508,13.378292,23205,155,0.234151,76.923077,0,0.0,0.003670,0.008427,0.049383


In [6]:
train.tail(10)

,u,v,cn,rai,jc,aai,pa,ccn,cra,wic,label,sim_atc,sim_mesh,sim_ade,sim_chemical
852728,DB00770,DB06335,14,0.035789,0.030837,2.324911,42435,14,0.000000,0.000000,0,0.0,0.000073,0.070720,0.072289
852729,DB04855,DB06770,266,0.533205,0.340589,42.602817,253460,266,0.000000,0.000000,1,0.0,0.000000,0.017122,0.074627
852730,DB01483,DB09279,173,0.432460,0.368870,28.618210,82305,173,0.000000,0.000000,1,0.0,0.027686,0.000000,0.125000
852731,DB04833,DB09265,0,0.000000,0.000000,0.000000,2331,0,0.000000,0.000000,0,0.0,0.000000,0.000000,0.080925
852732,DB00949,DB06212,176,0.315622,0.293823,27.734362,149974,176,0.000000,0.000000,0,0.0,0.000604,0.000000,0.112676
852733,DB06708,DB01143,58,0.108125,0.110476,9.187977,78810,58,0.000000,0.000000,0,0.0,0.029524,0.000000,0.058824
852734,DB00796,DB08807,129,0.331903,0.223958,21.488668,110214,129,0.000000,0.000000,0,1.0,0.044326,0.000000,0.173913
852735,DB00422,DB00692,150,0.417578,0.405405,25.193934,66700,270,0.367656,3.999867,1,0.0,0.042537,0.070479,0.134328
852736,DB00611,DB01406,5,0.009079,0.012165,0.788570,20764,5,0.000000,0.000000,0,0.0,0.019229,0.068817,0.170732
852737,DB00849,DB00807,265,0.656072,0.531062,43.908148,132468,513,0.633886,14.587377,1,0.0,0.029248,0.000000,0.131148


---

# 8. Chia dữ liệu tỉ lệ 1:1

In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

def balance_shuffle_and_split(input_csv, train_out, test_out, test_ratio=0.2):
    print(f"[1] Đang đọc dữ liệu từ: {input_csv}")
    try:
        df = pd.read_csv(input_csv)
        
        # 1. Tách riêng mẫu âm (0) và mẫu dương (1)
        df_pos = df[df['label'] == 1]
        df_neg = df[df['label'] == 0]
        
        num_pos = len(df_pos)
        num_neg = len(df_neg)
        
        print(f" -> Số lượng mẫu dương (1): {num_pos}")
        print(f" -> Số lượng mẫu âm (0):   {num_neg}")

        # 2. Lấy mẫu âm với số lượng bằng mẫu dương
        # Nếu mẫu âm đang nhiều hơn, ta lấy ngẫu nhiên một lượng bằng num_pos
        if num_neg > num_pos:
            print(f"\n[2] Đang thực hiện Under-sampling: Lấy {num_pos} mẫu âm để cân bằng...")
            df_neg_balanced = df_neg.sample(n=num_pos, random_state=42)
        else:
            print(f"\n[2] Cảnh báo: Mẫu âm đang ít hơn hoặc bằng mẫu dương. Giữ nguyên.")
            df_neg_balanced = df_neg

        # 3. Gộp lại thành bộ dữ liệu cân bằng (50/50)
        df_balanced = pd.concat([df_pos, df_neg_balanced], axis=0)
        
        print(f" -> Tổng số dòng sau khi cân bằng: {len(df_balanced)}")

        # 4. Trộn (Shuffle) và chia tỷ lệ 8/2
        print(f"\n[3] Đang trộn dữ liệu và chia tỷ lệ 8/2...")
        train_df, test_df = train_test_split(
            df_balanced, 
            test_size=test_ratio, 
            random_state=42, 
            shuffle=True, 
            stratify=df_balanced['label'] 
        )

        print(f" -> Kích thước tập Train (80%): {train_df.shape}")
        print(f" -> Kích thước tập Test (20%):  {test_df.shape}")

        # 5. Lưu ra file
        print(f"\n[4] Đang lưu file...")
        train_df.to_csv(train_out, index=False)
        test_df.to_csv(test_out, index=False)
        
        print(" -> HOÀN TẤT!")

    except Exception as e:
        print(f" Lỗi xử lý: {e}")

if __name__ == "__main__":
    INPUT_FILE = "/kaggle/input/datasets/vungocthien/dataset-dont-split/Final_Dataset_DDI.csv"     
    TRAIN_OUTPUT = "train_balanced_80.csv"
    TEST_OUTPUT = "test_balanced_20.csv"
    
    if os.path.exists(INPUT_FILE):
        balance_shuffle_and_split(INPUT_FILE, TRAIN_OUTPUT, TEST_OUTPUT)
    else:
        print(f"Lỗi: Không tìm thấy file '{INPUT_FILE}'.")

[1] Đang đọc dữ liệu từ: /kaggle/input/datasets/vungocthien/dataset-dont-split/Final_Dataset_DDI.csv
 -> Số lượng mẫu dương (1): 195150
 -> Số lượng mẫu âm (0):   1096878

[2] Đang thực hiện Under-sampling: Lấy 195150 mẫu âm để cân bằng...
 -> Tổng số dòng sau khi cân bằng: 390300

[3] Đang trộn dữ liệu và chia tỷ lệ 8/2...
 -> Kích thước tập Train (80%): (312240, 15)
 -> Kích thước tập Test (20%):  (78060, 15)

[4] Đang lưu file...
 -> HOÀN TẤT!


In [20]:
import pandas as pd
a = pd.read_csv('/kaggle/working/train_balanced_80.csv')
b = pd.read_csv('/kaggle/working/test_balanced_20.csv')

In [21]:
a.tail(50)

,u,v,cn,rai,jc,aai,pa,ccn,cra,wic,label,sim_atc,sim_mesh,sim_ade,sim_chemical
312190,DB11808,DB06480,34,0.075473,0.142857,5.488832,12871,55,0.042351,1.615260,0,0.000000,0.033590,0.000000,0.106383
312191,DB01224,DB09242,191,0.477620,0.263085,31.601719,163350,191,0.000000,0.000000,1,0.000000,0.015703,0.099199,0.081081
312192,DB08954,DB00162,1,0.013158,0.013514,0.230908,944,1,0.000000,0.000000,0,0.000000,0.002719,0.000000,0.083333
312193,DB00239,DB05109,52,0.095021,0.236364,8.172322,17127,97,0.083795,6.427653,0,0.000000,0.029381,0.010262,0.100840
312194,DB04743,DB00997,170,0.470235,0.228495,28.496033,205600,170,0.000000,0.000000,1,0.000000,0.021464,0.000000,0.104167
312195,DB00967,DB08987,265,0.656026,0.543033,43.907567,129542,513,0.633841,14.587377,1,0.000000,0.025994,0.000000,0.073529
312196,DB01225,DB00966,125,0.352200,0.249501,21.206322,80280,125,0.000000,0.000000,1,0.000000,0.023220,0.117234,0.084746
312197,DB01595,DB09320,5,0.011995,0.009980,0.826413,19065,5,0.000000,0.000000,0,0.000000,0.012099,0.000000,0.146067
312198,DB06216,DB01367,236,0.615328,0.336182,39.183019,218872,236,0.000000,0.000000,1,1.000000,0.066008,0.000000,0.177419
312199,DB11774,DB06782,3,0.041729,0.015789,0.701663,570,6,0.041729,3000.000000,0,0.000000,0.003802,0.000000,0.032787


In [22]:
b.tail(50)

,u,v,cn,rai,jc,aai,pa,ccn,cra,wic,label,sim_atc,sim_mesh,sim_ade,sim_chemical
78010,DB09063,DB00223,91,0.202950,0.113325,14.765332,136305,91,0.000000,0.000000,1,0.000000,0.001096,0.000000,0.078947
78011,DB03585,DB00218,133,0.423037,0.270876,22.971867,96768,251,0.393555,7.866142,1,0.526674,0.003597,0.000000,0.121951
78012,DB00588,DB09561,8,0.040538,0.015936,1.506622,9329,8,0.000000,0.000000,0,0.000000,0.000000,0.000000,0.048780
78013,DB06209,DB00997,118,0.199572,0.207018,18.430368,89436,215,0.158997,4.618828,1,0.000000,0.002883,0.096670,0.108108
78014,DB01263,DB01256,79,0.119313,0.125198,12.147924,50400,157,0.117849,77.922078,1,0.000000,0.004391,0.000000,0.125984
78015,DB01609,DB00281,366,1.185717,0.433649,61.681376,365496,366,0.000000,0.000000,1,0.000000,0.004843,0.000000,0.163636
78016,DB00988,DB11783,44,0.075554,0.089613,6.879385,59346,44,0.000000,0.000000,0,1.000000,0.055077,0.000000,0.126984
78017,DB06663,DB00678,121,0.261272,0.158793,19.583788,134652,121,0.000000,0.000000,0,0.000000,0.000076,0.000000,0.105691
78018,DB00419,DB00581,0,0.000000,0.000000,0.000000,100,0,0.000000,0.000000,0,1.000000,0.039078,0.050335,0.169811
78019,DB00712,DB01324,183,0.542032,0.314433,31.156251,143856,183,0.000000,0.000000,1,0.000000,0.007074,0.131418,0.125000


---